#Lab 11: Simulated Annealing and Constraint Satisfaction Problems

##Overview

In this lab, you will implement:

- Simulated Annealing for optimization
- Node Consistency for unary constraints
- Arc Consistency for binary constraints
- AC-3 algorithm for full constraint propagation

All algorithms must read from input files and produce outputs using a fixed OutputWriter.

##Input File Format

You will use two input files.

###Simulated Annealing Input File

Each line contains a parameter in key-value format:

key = value

Required parameters:
T (initial temperature)
cooling_rate
max_iterations
initial_state

###CSP Input File

This file contains two sections.

First section: variable domains

Format:
Variable: value1,value2,value3

Second section starts with:

constraints:

Each line after this defines a binary constraint:

X != Y

##Step 01: Make Imports

In [9]:
import random
import math
from collections import deque

##Step 02: Input Handler (Do NOT Modify)

In [10]:
class InputHandler:
    def __init__(self, path):
        self.path = path

    def load_sa(self):
        config = {}
        with open(self.path, "r") as f:
            for line in f:
                if "=" in line:
                    k, v = line.strip().split("=")
                    config[k.strip()] = float(v.strip())

        config["initial_state"] = int(config["initial_state"])
        config["max_iterations"] = int(config["max_iterations"])
        return config

    def load_csp(self):
        domains = {}
        constraints = []
        reading_constraints = False

        with open(self.path, "r") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue

                if line.lower().startswith("constraints"):
                    reading_constraints = True
                    continue

                if not reading_constraints:
                    var, dom = line.split(": ")
                    domains[var.strip()] = list(map(int, dom.split(",")))
                else:
                    constraints.append(tuple(line.split(" != ")))

        return domains, constraints

##Step 03: Simulated Annealing

In [11]:
class SimulatedAnnealing:
    def __init__(self, config):
        self.state = config["initial_state"]
        self.T = config["T"]
        self.cooling_rate = config["cooling_rate"]
        self.max_iterations = config["max_iterations"]

    def cost(self, x):
        return x*2   # example cost function (minimizing)

    def neighbor(self, x):
        return x + random.choice([-1, 1])

    def run(self):
        log = []

        current = self.state
        current_cost = self.cost(current)

        for i in range(self.max_iterations):
            next_state = self.neighbor(current)
            next_cost = self.cost(next_state)

            delta = next_cost - current_cost

            if delta < 0:
                current = next_state
                current_cost = next_cost
            else:
                prob = math.exp(-delta / self.T)
                if random.random() < prob:
                    current = next_state
                    current_cost = next_cost

            self.T *= self.cooling_rate

            log.append((i, current, current_cost, self.T))

        return log

##Step 04: Node Consistency

In [ ]:
class NodeConsistency:
    def __init__(self, domains):
        self.domains = domains

    def enforce(self):
        # removing duplicates
        for var in self.domains:
            self.domains[var] = list(set(self.domains[var]))
        return self.domains

##Step 05: Arc Consistency

In [13]:
class ArcConsistency:
    def __init__(self, domains, constraints):
        self.domains = domains
        self.constraints = constraints

    def revise(self, Xi, Xj):
        revised = False

        for x in self.domains[Xi][:]:
            if not any(x != y for y in self.domains[Xj]):
                self.domains[Xi].remove(x)
                revised = True

        return revised

    def enforce(self):
        for (Xi, Xj) in self.constraints:
            self.revise(Xi, Xj)
            self.revise(Xj, Xi)
        return self.domains

##Step 06: AC-3

In [14]:
class AC3:
    def __init__(self, domains, constraints):
        self.domains = domains
        self.constraints = constraints

    def revise(self, Xi, Xj):
        revised = False

        for x in self.domains[Xi][:]:
            if not any(x != y for y in self.domains[Xj]):
                self.domains[Xi].remove(x)
                revised = True

        return revised

    def ac3(self):
        queue = deque(self.constraints)

        for (Xi, Xj) in self.constraints:
            queue.append((Xj, Xi))

        while queue:
            Xi, Xj = queue.popleft()

            if self.revise(Xi, Xj):
                if not self.domains[Xi]:
                    return False

                for (Xk, Xl) in self.constraints:
                    if Xk == Xi and Xl != Xj:
                        queue.append((Xl, Xi))
                    elif Xl == Xi and Xk != Xj:
                        queue.append((Xk, Xi))

        return self.domains

##Step 07: Output Writer (Do NOT modify)

In [15]:
class OutputWriter:
    def __init__(self, path):
        self.path = path

    def write(self, title, data):
        with open(self.path, "a") as f:
            f.write("\n=== " + title + " ===\n")
            f.write(str(data) + "\n")

##Step 08: Main Method (Do NOT Modify)

In [16]:
def main():
    sa_config = InputHandler("sa_input.txt").load_sa()
    domains, constraints = InputHandler("csp_input.txt").load_csp()

    sa = SimulatedAnnealing(sa_config)
    sa_result = sa.run()

    nc = NodeConsistency(domains)
    nc_result = nc.enforce()

    ac = ArcConsistency(nc_result, constraints)
    ac_result = ac.enforce()

    ac3 = AC3(domains, constraints)
    ac3_result = ac3.ac3()

    writer = OutputWriter("output.txt")

    print(sa_result)
    print(nc_result)
    print(ac_result)
    print(ac3_result)

    writer.write("SA", sa_result)
    writer.write("NODE", nc_result)
    writer.write("ARC", ac_result)
    writer.write("AC3", ac3_result)


if __name__ == "__main__":
    main()

[(0, 38, 76, 232.5), (1, 39, 78, 216.22500000000002), (2, 38, 76, 201.08925000000002), (3, 39, 78, 187.01300250000003), (4, 40, 80, 173.92209232500002), (5, 41, 82, 161.74754586225004), (6, 42, 84, 150.42521765189255), (7, 41, 82, 139.89545241626007), (8, 42, 84, 130.10277074712187), (9, 41, 82, 120.99557679482334), (10, 40, 80, 112.52588641918571), (11, 39, 78, 104.64907436984272), (12, 38, 76, 97.32363916395374), (13, 39, 78, 90.51098442247698), (14, 40, 80, 84.17521551290359), (15, 41, 82, 78.28295042700034), (16, 42, 84, 72.80314389711032), (17, 41, 82, 67.7069238243126), (18, 42, 84, 62.967439156610716), (19, 43, 86, 58.55971841564797), (20, 42, 84, 54.46053812655261), (21, 41, 82, 50.648300457693935), (22, 40, 80, 47.102919425655365), (23, 39, 78, 43.805715065859495), (24, 38, 76, 40.73931501124933), (25, 39, 78, 37.88756296046188), (26, 38, 76, 35.23543355322955), (27, 39, 78, 32.76895320450348), (28, 40, 80, 30.47512648018824), (29, 41, 82, 28.341867626575066), (30, 40, 80, 26.